## Creating a Simple Foundry Agent with the SDK

### Installing Utilities and Libraries

In [ ]:
%pip install azure-ai-projects==2.7.0 openai==1.109.1 python-dotenv azure-identity

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

### Setting up the Foundry Project Client

In [ ]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Setting up our Batman Agent

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent_name = "batman-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are Batman, the Dark Knight of Gotham City. Respond to all queries in character as Batman would.",
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

### Creating a Conversation Object for the Agent Chat System

In [ ]:
# creating an openai client first
openai_client = project_client.get_openai_client()

# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

### Chat with the Agent

In [ ]:
chat = True

while chat:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        chat = False
        print("Exiting chat. Goodbye!")
    else:
        response = openai_client.responses.create(
            conversation=conversation.id,
            extra_body = {
                "agent": {
                    "name": agent_name,
                    "type": "agent_reference"
                }
            },
            input = user_input
        )

        print(f"Batman: {response.output_text}")

### Stream Responses from the Agent

In [ ]:
with openai_client.responses.create(
        conversation=conversation.id,
        extra_body = {
                        "agent": {
                            "name": agent_name,
                            "type": "agent_reference"
                        }
        },
        input = "How is Gotham City doing today?",
        stream=True,
    ) as response_stream_events:

        for event in response_stream_events:
            if event.type == "response.created":
                print(f"Stream response created with ID: {event.response.id}\n")
            elif event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "response.text.done":
                print("\n\nResponse text done. Access final text in 'event.text'")
            elif event.type == "response.completed":
                print("\n\nResponse completed. Access final text in 'event.response.output_text'")

### Send Image as Input with Streaming Responses

![batman_interrogation](./Images/batman_interrogation.png)

In [ ]:
image_url = "https://raw.githubusercontent.com/kuljotSB/AI-500-Multi-Agent-AI-Solutions/refs/heads/main/AI-Foundations/Images/batman_interrogation.png"

with openai_client.responses.create(
        conversation=conversation.id,
        extra_body = {
                        "agent": {
                            "name": agent_name,
                            "type": "agent_reference"
                        }
        },
        input=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Describe the attached image"
                    },
                    {
                        "type": "input_image",
                        "image_url": image_url
                    }
                ]
            }
        ],
        stream=True,
    ) as response_stream_events:

        for event in response_stream_events:
            if event.type == "response.created":
                print(f"Stream response created with ID: {event.response.id}\n")
            elif event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "response.text.done":
                print("\n\nResponse text done. Access final text in 'event.text'")
            elif event.type == "response.completed":
                print("\n\nResponse completed. Access final text in 'event.response.output_text'")